# Capstone build --- Chapter 12: Runtime Governance

Chapter~6 put a two-gate stack in front of the tools: a syntax check and the banking policies. Chapter~12 adds the third gate and the record. The third gate is the trained GMS plausibility gate, which scores each workflow transition against the banking store's learned dependency structure and refuses a step that does not follow from the last. The record is the audit log, a tamper-evident chain the harness writes as it runs, so every decision can be replayed and verified after the fact.

## The complete gate stack

`build_complaint_harness` assembles the governance layer: the registry, the policy engine and the plausibility gate, wrapped in a `GovernedToolExecutor` inside a `GovernanceHarness`. Reading the executor's gates shows the three-gate stack the shipped agent runs behind --- syntax, policy, plausibility.

In [ ]:
import json
from pathlib import Path
from forgeloop.agents.capstone import build_complaint_harness
from forgeloop.agents.gms_backend import GMSPlausibilityGate

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code'))
             if (c / 'data' / 'eval_cases' / 'cases.json').exists()), Path('.'))
harness, registry = build_complaint_harness(policies_dir=root / 'data' / 'policies')
gates = harness._executor._gates
for g in gates:
    print(f'{g.__class__.__name__:22s} trained={isinstance(g, GMSPlausibilityGate)}')

The plausibility gate is the one component the reader does not build by hand: it is loaded from the calibrated GMS banking store, and its threshold is the one persisted at calibration time. This is the build-not-train boundary of the series --- the geometry is trained and imported, the stack it sits in is assembled here.

## Governance produces an audit chain

Running a case through the harness logs an event per step. The audit log hashes each state and links the events, so `verify` can confirm the chain was not altered. This is what makes a run accountable: the sequence of decisions is recorded, ordered and checkable.

In [ ]:
from forgeloop.agents.core import Budget, BudgetTracker, TaskSpec

cases = json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())
case = next(c for c in cases if c['id'] == 'case-002')
task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
print('final status     :', traj.final_state.status)
print('audit events     :', len(harness.audit.events))
print('audit chain valid:', harness.audit.verify())

## A gate refusal is recorded, not silent

When a gate denies a call the harness records a failed step; the agent reads that failure and escalates. The governance layer never drops a decision silently --- a refusal is an auditable event like any other. The audit log below shows the ordered statuses the run passed through.

In [ ]:
for rec in traj.records:
    print(f'step {rec.step}: {rec.action.kind:10s} -> {rec.state_after.status}')

Runtime governance is the gate stack and the audit chain together: the stack decides what may run, and the chain records what did. The harness assembled here is the object `build_complaint_harness` returns; Chapter~14 traces the escalation paths out of it, and Chapter~16 assembles the whole capstone and checks its wiring against the shipped harness.